<h1> Bertopic</h1>

In this notebook, we will be running bertopic again on our collected data. We will try to get as many topics as possible in only on run since it is good for the topic alighment with other platforms. 

<h2> Data prep </h2>

We will upload already ready data. We will delete the unrelavant videos with the help of the 'is_relevant' column that we have created in the previous analysis of the Topics and Subtopics. 

In [1]:
import pandas as pd

df = pd.read_csv("./topics/subtopics/final_data.csv")

In [10]:
pd.set_option('display.max_rows', None)

In [3]:
df.columns

Index(['source', 'video_id', 'video_url', 'title', 'description', 'channel',
       'comments', 'views', 'likes', 'comments_count', 'keywords', 'country',
       'date', 'likes_missing', 'text', 'topic', 'is_relevant', 'subtopic'],
      dtype='object')

In [4]:
#let's only get the relevant topics
df = df[df['is_relevant']==True]

In [5]:
df.shape

(15087, 18)

In [6]:
#Let's drop the columns 'topic', 'is_relvant', 'subtopic' and 'text'
df = df.drop(columns=['topic', 'is_relevant', 'subtopic', 'text'])
df.columns

Index(['source', 'video_id', 'video_url', 'title', 'description', 'channel',
       'comments', 'views', 'likes', 'comments_count', 'keywords', 'country',
       'date', 'likes_missing'],
      dtype='object')

In [7]:
#Let's create te 'text' column again
df['text'] = df['title'].fillna("") + '. ' + df['description'].fillna("")

In [8]:
#Let's create a function which will clean the text from unwanted words

import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # remove URLs
    text = re.sub(r"www\S+", "", text)       # remove www links
    text = re.sub(r"\byoutube\b", "", text)  # remove "youtube"
    text = re.sub(r"\bvideo\b", "", text)    # remove "video"
    text = re.sub(r"\bwatch\b", "", text)    # remove "watch"
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # keep only words
    return text.lower().strip()

In [9]:
df["clean_text"] = df["text"].fillna("").map(clean_text)

In [10]:
df.head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text
0,YouTube,Mp6qdlHJyyI,https://www.youtube.com/watch?v=Mp6qdlHJyyI,Food for Seniors,Food for seniors is an important concern. Food...,GracefulAging,NaN,661.0,1.0,0.0,healthy aging,unknown,2010-01-03,False,Food for Seniors. Food for seniors is an impor...,food for seniors food for seniors is an impor...
1,YouTube,FxTc69tuOBA,https://www.youtube.com/watch?v=FxTc69tuOBA,Joe Rogan ~ DMT Is A Portal To The After Life,Joe Rogan explains his experience with DMT (Di...,Alistar Valadez,Great channel. After 5Meo I stopped lookung at...,144742.0,889.0,288.0,nootropics,unknown,2010-01-03,False,Joe Rogan ~ DMT Is A Portal To The After Life....,joe rogan dmt is a portal to the after life ...
2,YouTube,bWrlSE-LDdc,https://www.youtube.com/watch?v=bWrlSE-LDdc,THE KUROTEL SPA AND LONGEVITY CENTER,THE KUROTEL SPA AND LONGEVITY CENTER With an i...,Better Living - Travel. Food. Home. Lifestyle.,NaN,1067.0,2.0,0.0,longevity,en,2010-01-03,False,THE KUROTEL SPA AND LONGEVITY CENTER. THE KURO...,the kurotel spa and longevity center the kuro...
3,YouTube,EgOeF0YZu-A,https://www.youtube.com/watch?v=EgOeF0YZu-A,Coconut Kefir Margarita @ Longevity Conference...,Follow J-Wro on these sites (scroll down for m...,Jason Wrobel,NOTE TO ALL FANS OF DAVID AVOCADO WOLFE: Avoca...,2469.0,9.0,2.0,longevity,unknown,2010-01-03,False,Coconut Kefir Margarita @ Longevity Conference...,coconut kefir margarita longevity conference...
4,YouTube,7VVTsotiIms,https://www.youtube.com/watch?v=7VVTsotiIms,Qi Gong 10 symbols of Longevity Exercises 1-6,This rarely known in America set of Qi Gong Ex...,Luis Duarte Tai Chi & Qi Gong,"💖👏👏👏🙏🇵🇹 || Graciad profesor luis || Que Joven,...",401312.0,2249.0,191.0,longevity,en-US,2010-01-03,False,Qi Gong 10 symbols of Longevity Exercises 1-6....,qi gong symbols of longevity exercises ...


<h2> Bertopic run</h2>

In [11]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/azim/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [11]:
import umap
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Embedding model
model = SentenceTransformer("paraphrase-mpnet-base-v2")

# Custom stopwords
custom_stopwords = list(stopwords.words("english")) + ["http", "https", "youtube", "video", "videos", "watch", "channel", "subscribe", "com"]

# Vectorizer
vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1, 2),
    max_df=0.85,
    min_df=2,
    max_features=3000
)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    random_state=42
)

# HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=5,
    cluster_selection_epsilon=0.0,
    prediction_data=True
)

# BERTopic
topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    nr_topics=None,
    top_n_words=10,
    calculate_probabilities=True,
)

In [12]:
topics, probs = topic_model.fit_transform(df['clean_text'].fillna("").tolist())

In [13]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5266,-1_anti_anti aging_healthy_healthy aging,"[anti, anti aging, healthy, healthy aging, ski...",[anti aging skin care products anti aging ski...
1,0,1875,0_biohacking_biohacker_biohack_dave,"[biohacking, biohacker, biohack, dave, asprey,...",[biohacking is amazing biohacking...
2,1,899,1_longevity_human longevity_human_live,"[longevity, human longevity, human, live, long...","[lesson longevity, pcf longevity pcf long..."
3,2,427,2_facial rejuvenation_facial_plastic_surgery,"[facial rejuvenation, facial, plastic, surgery...","[nyt alison facial rejuvenation, aiello ..."
4,3,391,3_exercise_fitness_training_fit,"[exercise, fitness, training, fit, exercises, ...",[fit body fit mind your practical guide to ag...
5,4,374,4_brain_focus_nootropics_memory,"[brain, focus, nootropics, memory, boost, supp...",[how to boost brain power nootropics supplem...
6,5,235,5_brain_alzheimer_dementia_aging brain,"[brain, alzheimer, dementia, aging brain, brai...",[how much sleep do i need brain rules for ag...
7,6,217,6_nutrition_eating_aging nutrition_diet,"[nutrition, eating, aging nutrition, diet, foo...","[aging and nutrition, iff healthy aging nutr..."
8,7,191,7_laser_laser skin_skin rejuvenation_skin,"[laser, laser skin, skin rejuvenation, skin, r...",[laser skin rejuvenation visit laser skin ...
9,8,191,8_cortex_cortex nootropic_stack torque_socials...,"[cortex, cortex nootropic, stack torque, socia...",[these nootropics are all you need our pro...


<h3> Saving </h3>

In [15]:
tpp = topic_model.get_topic_info()

In [17]:
import numpy as np
import pandas as pd
import pickle


tpp.to_csv('./topics_2/topics_before/mpnet_topics.csv', index=False)
topic_model.save("./topics_2/topics/mpnet_topic_model")
np.save('./topics_2/topics/topic_probabilities.npy', probs)

preprocessed_data = df['clean_text'].fillna("").tolist()
with open('./topics_2/topics/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(preprocessed_data, f)

2025-09-03 02:53:51,824 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [18]:
st = SentenceTransformer("paraphrase-mpnet-base-v2")
def embed(texts):
    return st.encode(texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True)

X = embed(df['clean_text'].fillna("").tolist()).astype("float32")
np.save("./topics_2/topics/embeddings.npy", X)

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

In [20]:
#Saving without UMAP
topic_model.umap_model = None
topic_model.save("./topics_2/topics/mpnet_topic_model_no_umap")

2025-09-03 03:12:43,491 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [21]:
#Assigning the topics
df['topic']=topics

#saving the data
df.to_csv("./topics_2/topics_before/data.csv", index=False)

<h2> Bertopics analysis </h2>

In [2]:
import pandas as pd

df = pd.read_csv("./topics_2/topics_before/data.csv")
tp = pd.read_csv("./topics_2/topics_before/mpnet_topics.csv")

In [12]:
#function to check if the video is related or unrelated

def is_video_related(text, keywords):
    text = str(text).lower()
    return any(kw in text for kw in keywords)

def check_relevance_ratio(df, topic_id, keywords):
    df_topic = df[df['topic']==topic_id]
    relevant_count = df_topic['clean_text'].apply(lambda x: is_video_related(x, keywords)).sum()
    return relevant_count/len(df_topic)

keywords = ["longevity", "healthy aging", "biohacking", "rejuvenation", "anti-aging", "nootropics", "health", "well-aging",
            "aging well", "aging"]

tp['relevance_ratio'] = tp['Topic'].apply(lambda t: check_relevance_ratio(df, t, keywords)) 

In [13]:
tp

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
0,-1,5266,-1_anti_anti aging_healthy_healthy aging,"['anti', 'anti aging', 'healthy', 'healthy agi...",['anti aging skin care products anti aging sk...,0.885302
1,0,1875,0_biohacking_biohacker_biohack_dave,"['biohacking', 'biohacker', 'biohack', 'dave',...",['biohacking is amazing biohackin...,0.782933
2,1,899,1_longevity_human longevity_human_live,"['longevity', 'human longevity', 'human', 'liv...","['lesson longevity', 'pcf longevity pcf l...",0.975528
3,2,427,2_facial rejuvenation_facial_plastic_surgery,"['facial rejuvenation', 'facial', 'plastic', '...","['nyt alison facial rejuvenation', 'aiello...",0.957845
4,3,391,3_exercise_fitness_training_fit,"['exercise', 'fitness', 'training', 'fit', 'ex...",['fit body fit mind your practical guide to a...,0.915601
5,4,374,4_brain_focus_nootropics_memory,"['brain', 'focus', 'nootropics', 'memory', 'bo...",['how to boost brain power nootropics supple...,0.745989
6,5,235,5_brain_alzheimer_dementia_aging brain,"['brain', 'alzheimer', 'dementia', 'aging brai...",['how much sleep do i need brain rules for a...,0.914894
7,6,217,6_nutrition_eating_aging nutrition_diet,"['nutrition', 'eating', 'aging nutrition', 'di...","['aging and nutrition', 'iff healthy aging n...",0.935484
8,7,191,7_laser_laser skin_skin rejuvenation_skin,"['laser', 'laser skin', 'skin rejuvenation', '...",['laser skin rejuvenation visit laser skin...,0.947644
9,8,191,8_cortex_cortex nootropic_stack torque_socials...,"['cortex', 'cortex nootropic', 'stack torque',...",['these nootropics are all you need our pr...,0.958115


Let's check the topics with less than the 40 percent releavance. 

In [16]:
tp[tp['relevance_ratio']<0.40]

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
15,14,125,14_drugs_smart drugs_smart_students,"['drugs', 'smart drugs', 'smart', 'students', ...","['scienziati della droga smart drugs', 'smar...",0.104000
60,59,38,59_alpha brain_alpha_onnit_brain,"['alpha brain', 'alpha', 'onnit', 'brain', 'jo...",['alpha brain review real user experience onni...,0.289474
61,60,38,60_noopept_nootropic benefits_nootropic_including,"['noopept', 'nootropic benefits', 'nootropic',...",['valerian in this you ll discover the nootr...,0.289474
81,80,27,80_big think_edge exclusive_daily join_new daily,"['big think', 'edge exclusive', 'daily join', ...",['what s your counsel new videos daily ...,0.296296
92,91,22,91_neurofeedback_therapy_orange_healing,"['neurofeedback', 'therapy', 'orange', 'healin...",['is neurofeedback therapy a useful therapy ...,0.090909
105,104,21,104_aged_aged well_surgeon_reacts,"['aged', 'aged well', 'surgeon', 'reacts', 'do...",['has erika jayne aged well surgeon reacts ...,0.238095


We got only 6 topics. However, even though the percentage is smaller than the 40 percent, it seems like they are more or less related. We can
check the top five videos for each topics and decide if they are related or not

In [17]:
#Topic number 14
df[df['topic']==14].head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
1,YouTube,FxTc69tuOBA,https://www.youtube.com/watch?v=FxTc69tuOBA,Joe Rogan ~ DMT Is A Portal To The After Life,Joe Rogan explains his experience with DMT (Di...,Alistar Valadez,Great channel. After 5Meo I stopped lookung at...,144742.0,889.0,288.0,nootropics,unknown,2010-01-03,False,Joe Rogan ~ DMT Is A Portal To The After Life....,joe rogan dmt is a portal to the after life ...,14
166,YouTube,nz9piyDdfrU,https://www.youtube.com/watch?v=nz9piyDdfrU,Misuse of Common Drugs Can Be Deadly,Common over-the-counter drugs can cause danger...,CBS,LOLZ,880.0,2.0,1.0,nootropics,unknown,2010-03-23,False,Misuse of Common Drugs Can Be Deadly. Common o...,misuse of common drugs can be deadly common o...,14
218,YouTube,CSj_FdOQACw,https://www.youtube.com/watch?v=CSj_FdOQACw,Excerpt: Boosting Brain Power,"More people, especially college students, are ...",CBS,Keep telling people you were on it. They might...,13596.0,13.0,6.0,nootropics,unknown,2010-04-22,False,"Excerpt: Boosting Brain Power. More people, es...",excerpt boosting brain power more people es...,14
219,YouTube,f_wbAhVK5eA,https://www.youtube.com/watch?v=f_wbAhVK5eA,Excerpt: Boosting Brain Power,"More people, especially college students, are ...",CBS News,Wow just Wow,1198.0,3.0,1.0,nootropics,unknown,2010-04-22,False,"Excerpt: Boosting Brain Power. More people, es...",excerpt boosting brain power more people es...,14
221,YouTube,xOlNo3ECuZI,https://www.youtube.com/watch?v=xOlNo3ECuZI,Extra: Adderall &amp; Academic Achievement,"Is there pressure to take ""smart drugs"" to sta...",CBS,i&#39;ve been taking adderall for like year an...,2952.0,3.0,5.0,nootropics,en,2010-04-25,False,Extra: Adderall &amp; Academic Achievement. Is...,extra adderall amp academic achievement is...,14


Videos in this topic are related to our research. It talks about the nootropics which help to boost the abilites of the brain

In [18]:
#topic number 59
df[df['topic']==59].head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
1844,YouTube,FNjXd6cAmbE,https://www.youtube.com/watch?v=FNjXd6cAmbE,New Mood™ From Onnit Labs On the Joe Rogan Exp...,"New Mood™ from Onnit Labs, is discussed by CEO...",Onnit,GOD BLESS Joe Rogan and Onnit...you dont have ...,74258.0,365.0,96.0,nootropics,unknown,2012-03-02,False,New Mood™ From Onnit Labs On the Joe Rogan Exp...,new mood from onnit labs on the joe rogan exp...,59
2023,YouTube,jkAzqZGM8TA,https://www.youtube.com/watch?v=jkAzqZGM8TA,Onnit Labs Alpha Brain and New Mood Review,My personal review of Onnit Labs Alpha Brain a...,Christian Golez,Ya apparently doesn’t work with ums. Not a goo...,16553.0,57.0,39.0,nootropics,en,2012-04-26,False,Onnit Labs Alpha Brain and New Mood Review. M...,onnit labs alpha brain and new mood review m...,59
2031,YouTube,zJGCMatfel8,https://www.youtube.com/watch?v=zJGCMatfel8,AlphaBrain - ноотроп нового поколения,Промо - видео революционного продукта Onnit La...,Onnit Russia,"На западе ноотропы очень популярны , и востреб...",11352.0,43.0,7.0,nootropics,en,2012-05-02,False,AlphaBrain - ноотроп нового поколения. Промо -...,alphabrain ...,59
2158,YouTube,lfeO3DoxBME,https://www.youtube.com/watch?v=lfeO3DoxBME,Alpha Brain review part 2 and Thank Yous,Just a quick reply to everyone that's helped p...,SixStringPirate,"Your dreams sound fucking hilarious, ha ha! Ge...",1848.0,14.0,18.0,nootropics,unknown,2012-06-09,False,Alpha Brain review part 2 and Thank Yous. Just...,alpha brain review part and thank yous just...,59
2403,YouTube,6vh00ELP-Zk,https://www.youtube.com/watch?v=6vh00ELP-Zk,Alphabrain shroomtech sport end of day one,Day one still but after it!!,Chris Wellington,I&#39;ve just had two alpha and two shroom. F...,466.0,3.0,3.0,nootropics,en,2012-09-21,False,Alphabrain shroomtech sport end of day one. Da...,alphabrain shroomtech sport end of day one da...,59


This topic is about the nootropic called alpha brain

In [19]:
#topic number 60
df[df['topic']==60].head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
2036,YouTube,_LvkAHbq7NI,https://www.youtube.com/watch?v=_LvkAHbq7NI,Nootropics Series - Part 1: Nootropic Drugs an...,The aim of this lecture entitled 'The role of ...,AntiagingSystems,I take 10mg × 2 a day. It&#39;s a brain fog ki...,52592.0,374.0,41.0,nootropics,en,2012-05-04,False,Nootropics Series - Part 1: Nootropic Drugs an...,nootropics series part nootropic drugs an...,60
2439,YouTube,Tm40c2s4YK8,https://www.youtube.com/watch?v=Tm40c2s4YK8,My Thoughts on Noopept....and Alphabrain,I am currently taking noopept and alphabrain o...,anthonynlee,"Keep them coming, enjoyed a lot! || you take 1...",43445.0,273.0,133.0,nootropics,en,2012-10-06,False,My Thoughts on Noopept....and Alphabrain. I am...,my thoughts on noopept and alphabrain i am...,60
2931,YouTube,LNdk58e0RzQ,https://www.youtube.com/watch?v=LNdk58e0RzQ,My Noopept Review,Here's my youtube video review on Noopept - a ...,po3tik1,I had to take 40 mg (x2 doses) to get the effe...,14942.0,50.0,20.0,nootropics,en,2013-04-25,False,My Noopept Review. Here's my youtube video rev...,my noopept review here s my review on noope...,60
3160,YouTube,aqF7ZCQO6Cw,https://www.youtube.com/watch?v=aqF7ZCQO6Cw,[Review] LiftMode Noopept Review,This is my review of LiftMode's 5g Noopept. To...,SupplementReviews8946,You only took it a few days so dont know how y...,32088.0,210.0,59.0,nootropics,unknown,2013-07-29,False,[Review] LiftMode Noopept Review. This is my r...,review liftmode noopept review this is my re...,60
3254,YouTube,E3OZh6dxxYw,https://www.youtube.com/watch?v=E3OZh6dxxYw,[UPDATE] Liftmode Noopept Update 1,This is a follow-up update on Liftmode's Noopept.,SupplementReviews8946,I love ANKI || What is you&#39;re dose. I just...,11705.0,61.0,39.0,nootropics,unknown,2013-09-09,False,[UPDATE] Liftmode Noopept Update 1. This is a ...,update liftmode noopept update this is a f...,60


This topic contains the videos about the review of the nootropics

In [20]:
#topic number 80
df[df['topic']==80].head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
1226,YouTube,DV3XjqW_xgU,https://www.youtube.com/watch?v=DV3XjqW_xgU,Michio Kaku: How to Reverse Aging | Big Think,New videos DAILY: https://bigth.ink Join Big T...,Big Think,"Want to get Smarter, Faster™?\r<br>Subscribe f...",2120423.0,25049.0,4264.0,well aging,en-US,2011-05-31,False,Michio Kaku: How to Reverse Aging | Big Think....,michio kaku how to reverse aging big think ...,80
1233,YouTube,5MeWBlKYOvw,https://www.youtube.com/watch?v=5MeWBlKYOvw,Neil deGrasse Tyson: Living and Longevity,New videos DAILY: https://bigth.ink Join Big T...,Big Think,Limited psychology about human potential. || I...,160718.0,2227.0,338.0,longevity,en-US,2011-06-03,False,Neil deGrasse Tyson: Living and Longevity. New...,neil degrasse tyson living and longevity new...,80
1248,YouTube,X57lJXW0zL0,https://www.youtube.com/watch?v=X57lJXW0zL0,What Is Evil?,New videos DAILY: https://bigth.ink Join Big T...,Big Think,Evil is repeated sadistic behaviour by usually...,21276.0,221.0,32.0,biohacking,en-US,2011-06-10,False,What Is Evil?. New videos DAILY: https://bigth...,what is evil new videos daily join big thi...,80
1265,YouTube,HjJYvLH_FGw,https://www.youtube.com/watch?v=HjJYvLH_FGw,The Neuroscience of Internet Addiction,New videos DAILY: https://bigth.ink Join Big T...,Big Think,It&#39;s so weird to watch this 13 years later...,120396.0,1491.0,188.0,nootropics,en-US,2011-06-20,False,The Neuroscience of Internet Addiction. New vi...,the neuroscience of internet addiction new vi...,80
1989,YouTube,HgywkydIil8,https://www.youtube.com/watch?v=HgywkydIil8,What is your counsel?,New videos DAILY: https://bigth.ink Join Big T...,Big Think,NaN,138.0,0.0,0.0,biohacking,en-US,2012-04-20,False,What is your counsel?. New videos DAILY: https...,what is your counsel new videos daily join...,80


This topic contains the vidoes of the channel called Big Think. Big Think is a YouTube channel and platform featuring interviews and educational content from world-renowned experts and thinkers, designed to help viewers "get smarter, faster" by exploring big ideas and essential skills for the 21st century.

In [21]:
#topic number 91
df[df['topic']==91].head()

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
57,YouTube,LI2319Vcs_M,https://www.youtube.com/watch?v=LI2319Vcs_M,Author of Neurofeedback Book discussing &#39;Z...,Dr. Clare Albright is a psychologist and the a...,C Albright,NaN,2431.0,0.0,0.0,nootropics,en,2010-01-29,False,Author of Neurofeedback Book discussing &#39;Z...,author of neurofeedback book discussing z...,91
68,YouTube,RPrPL6WJ4ls,https://www.youtube.com/watch?v=RPrPL6WJ4ls,Holistic Brain Neurotherapy: Retrain Your Brain,"For Beyond 50's ""Natural Healing"" talks, watch...",Daniel Davis,&quot;You have too many high frequencies in th...,36342.0,137.0,7.0,nootropics,unknown,2010-02-03,False,Holistic Brain Neurotherapy: Retrain Your Brai...,holistic brain neurotherapy retrain your brai...,91
246,YouTube,XtnNpUyK3KM,https://www.youtube.com/watch?v=XtnNpUyK3KM,Live The Life You Love--Neuro-Enhancement Stra...,Information video on Neuro-Enhancement Strateg...,J Schoener,NaN,1391.0,0.0,0.0,nootropics,unknown,2010-05-03,False,Live The Life You Love--Neuro-Enhancement Stra...,live the life you love neuro enhancement stra...,91
312,YouTube,8fwgr1M0GX4,https://www.youtube.com/watch?v=8fwgr1M0GX4,Neurofeedback Therapy: Anxiety &amp; Panic Dis...,Please visit this site: www.neurofeedbackbook....,C Albright,NaN,18482.0,50.0,0.0,nootropics,en,2010-06-03,False,Neurofeedback Therapy: Anxiety &amp; Panic Dis...,neurofeedback therapy anxiety amp panic dis...,91
449,YouTube,jjY9ILDDOaM,https://www.youtube.com/watch?v=jjY9ILDDOaM,Depression and Neurofeedback,DoctorClarity's shared video file.,C Albright,this video will make you depressed || The musi...,4567.0,8.0,6.0,nootropics,unknown,2010-07-28,False,Depression and Neurofeedback. DoctorClarity's ...,depression and neurofeedback doctorclarity s ...,91


This video talks about the neuroscience

In [25]:
#topic number 104
df[df['topic']==104]

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,clean_text,topic
10469,YouTube,orioiI25J2w,https://www.youtube.com/watch?v=orioiI25J2w,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,NaN,Base Beauty Creative Agency,NaN,276.0,2.0,0.0,well aging,en,2021-03-10,False,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,bbca social ad pause well aging comedic clip,104
10472,YouTube,hpDWWYGhWZE,https://www.youtube.com/watch?v=hpDWWYGhWZE,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,NaN,Base Beauty Creative Agency,NaN,325.0,2.0,0.0,well aging,en,2021-03-10,False,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,bbca social ad pause well aging comedic clip,104
10473,YouTube,Q4Ay77IiTC8,https://www.youtube.com/watch?v=Q4Ay77IiTC8,BBCA Social Ad | Pause-Well Aging - Comedy Clip 6,NaN,Base Beauty Creative Agency,NaN,42.0,0.0,0.0,well aging,en,2021-03-10,False,BBCA Social Ad | Pause-Well Aging - Comedy Cli...,bbca social ad pause well aging comedy clip,104
10475,YouTube,MuS_0eiy--Y,https://www.youtube.com/watch?v=MuS_0eiy--Y,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,NaN,Base Beauty Creative Agency,NaN,32.0,0.0,0.0,well aging,en,2021-03-10,False,BBCA Social Ad | Pause Well-Aging - Comedic Cl...,bbca social ad pause well aging comedic clip,104
10476,YouTube,4gl0NcvyS4c,https://www.youtube.com/watch?v=4gl0NcvyS4c,BBCA Social Ad | Pause-Well Aging - Comedy Clip 4,NaN,Base Beauty Creative Agency,NaN,77.0,0.0,0.0,well aging,en,2021-03-10,False,BBCA Social Ad | Pause-Well Aging - Comedy Cli...,bbca social ad pause well aging comedy clip,104
13456,YouTube,Vqf_4yPV4Yw,https://www.youtube.com/watch?v=Vqf_4yPV4Yw,⏰Has Jane Fonda Aged Well? (Surgeon Reacts),Double Board Certified Plastic Surgeon Reviews...,Tansavatdi Facial Plastic Surgery,Nope still a traitor. || Beautiful || She look...,41813.0,1057.0,67.0,well aging,en-US,2024-04-22,False,⏰Has Jane Fonda Aged Well? (Surgeon Reacts). D...,has jane fonda aged well surgeon reacts do...,104
13474,YouTube,hD2bikurgJ8,https://www.youtube.com/watch?v=hD2bikurgJ8,Has Dolly Parton Aged Well? (Surgeon Reacts),Double Board Certified Plastic Surgeon Reviews...,Tansavatdi Facial Plastic Surgery,😂😂😂how on earth that ppl think she aged well??...,10461.0,362.0,18.0,well aging,en-US,2024-04-29,False,Has Dolly Parton Aged Well? (Surgeon Reacts). ...,has dolly parton aged well surgeon reacts ...,104
13507,YouTube,IVBi91HFIDU,https://www.youtube.com/watch?v=IVBi91HFIDU,⏰ Has Meg Ryan Aged Well? (Surgeon Reacts),Double Board Certified Plastic Surgeon Reviews...,Tansavatdi Facial Plastic Surgery,She doesn&#39;t look like herself || I&#39;m s...,22225.0,522.0,22.0,well aging,en-US,2024-05-13,False,⏰ Has Meg Ryan Aged Well? (Surgeon Reacts). Do...,has meg ryan aged well surgeon reacts doub...,104
13520,YouTube,K27sG5cWkZc,https://www.youtube.com/watch?v=K27sG5cWkZc,⏰ Has Simon Cowell Aged Well? (Surgeon Reacts),Double Board Certified Plastic Surgeon Reviews...,Tansavatdi Facial Plastic Surgery,It&#39;s a no from me. || Let me just say this...,41371.0,686.0,76.0,well aging,en-US,2024-05-20,False,⏰ Has Simon Cowell Aged Well? (Surgeon Reacts)...,has simon cowell aged well surgeon reacts ...,104
13579,YouTube,gT4udUrXdec,https://www.youtube.com/watch?v=gT4udUrXdec,⏰ Has Courteney Cox Aged Well? (Surgeon Reacts),Double Board Certified Plastic Surgeon Reviews...,Tansavatdi Facial Plastic Surgery,She should cut her hair. Long dark hair often ...,103879.0,2094.0,103.0,well aging,en-US,2024-06-13,False,⏰ Has Courteney Cox Aged Well? (Surgeon Reacts...,has courteney cox aged well surgeon reacts ...,104


It contains the some suspicious vidoes. However, the half of the vidoes are about whether the specific celecbrity aged well or not. It might be related. Because it might contain discussion about  people's perception of well aging

In summary, we can say that every topic was more or less related

In [26]:
import pandas as pd

df.to_csv("./topics_2/topics/data.csv", index=False)
tp.to_csv("./topics_2/topics/mpnet_topics.csv", index=False)

<h1> Next steps</h1>

**Next steps for the bertopic analysis**
- Try to deal with the noisy topics (later)